# Catalog and table setup
Creates the Unity Catalog schemas (`bronze`, `silver`, `gold`, `ops`) and initializes empty Delta tables for the pipeline.

### Key design choices
* **Dynamic catalog (`SELECT current_catalog()`):** Avoids hardcoding the catalog name so the same code can run in dev, test, or prod without changes.
* **Unity Catalog schemas (`bronze`, `silver`, `gold`, `ops`):** Separates each layer into its own schema to keep data organized and make access control simple (for example, giving dashboard users read access only to `gold`).
* **Shared Bronze table structure (`CREATE TABLE ... LIKE`):** Both Bronze tables use the exact same columns (raw JSON string, HTTP status, SHA-256 hashes, and run metadata), so they can share the same ingestion and merge logic.
* **Explicit schema rules:** Key columns and timestamps use `NOT NULL`, while weather measurements in Silver allow `NULL` values so missing sensor data is not filled with guessed values.

In [0]:
from pyspark.sql import functions as F

# Get the active catalog dynamically instead of hardcoding a name
spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

# Create schemas for each layer if they do not exist yet
for schema_name in ["bronze", "silver", "gold", "ops"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}")

# Bronze tables (`electricity_price_api_raw` and `weather_api_raw`): Stores one row per day with the raw JSON response and SHA-256 hashes (`bronze_record_id`) to prevent duplicates when re-running the job.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.electricity_price_api_raw (
    bronze_record_id STRING NOT NULL,
    source_system STRING NOT NULL,
    request_key STRING NOT NULL,
    request_date DATE NOT NULL,
    price_area STRING,
    station_id STRING,
    source_url STRING NOT NULL,
    http_status INT NOT NULL,
    payload_json STRING,
    payload_hash STRING NOT NULL,
    record_count INT NOT NULL,
    error_message STRING,
    retrieved_at_utc TIMESTAMP NOT NULL,
    ingestion_run_id STRING NOT NULL
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.weather_api_raw
LIKE {CATALOG}.bronze.electricity_price_api_raw
""")

# Silver tables (`electricity_price_hourly`, `weather_hourly`, `weather_price_hourly`): Stores cleaned hourly data converted to UTC (`time_start_utc`), along with lineage columns (`source_bronze_record_id`, `run_id`).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.electricity_price_hourly (
    price_area STRING NOT NULL,
    time_start_utc TIMESTAMP NOT NULL,
    time_end_utc TIMESTAMP,
    time_start_local TIMESTAMP,
    nok_per_kwh DOUBLE,
    eur_per_kwh DOUBLE,
    exchange_rate DOUBLE,
    source_bronze_record_id STRING,
    source_retrieved_at_utc TIMESTAMP,
    processed_at_utc TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.weather_hourly (
    station_id STRING NOT NULL,
    time_start_utc TIMESTAMP NOT NULL,
    time_start_local TIMESTAMP,
    air_temperature_c DOUBLE,
    wind_speed_mps DOUBLE,
    precipitation_1h_mm DOUBLE,
    temperature_quality_code INT,
    wind_quality_code INT,
    precipitation_quality_code INT,
    source_bronze_record_id STRING,
    source_retrieved_at_utc TIMESTAMP,
    processed_at_utc TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.weather_price_hourly (
    price_area STRING NOT NULL,
    station_id STRING NOT NULL,
    time_start_utc TIMESTAMP NOT NULL,
    time_end_utc TIMESTAMP,
    time_start_local TIMESTAMP,
    nok_per_kwh DOUBLE,
    eur_per_kwh DOUBLE,
    exchange_rate DOUBLE,
    air_temperature_c DOUBLE,
    wind_speed_mps DOUBLE,
    precipitation_1h_mm DOUBLE,
    weather_match_status STRING,
    processed_at_utc TIMESTAMP
) USING DELTA
""")

# Gold (`price_by_temperature_band`): Stores electricity prices grouped into 5°C temperature bands, including both mean and median price.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.gold.price_by_temperature_band (
    price_area STRING NOT NULL,
    temperature_band_order INT NOT NULL,
    temperature_band STRING NOT NULL,
    observation_count BIGINT,
    average_price_nok_per_kwh DOUBLE,
    median_price_nok_per_kwh DOUBLE,
    minimum_price_nok_per_kwh DOUBLE,
    maximum_price_nok_per_kwh DOUBLE,
    average_wind_speed_mps DOUBLE,
    updated_at_utc TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.gold.model_feature_importance (
    feature STRING,
    importance DOUBLE
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.gold.model_performance_metrics (
    mae DOUBLE,
    rmse DOUBLE,
    r2 DOUBLE
) USING DELTA
""")

# Ops (`data_quality_results`): Logs all data quality checks (`ERROR` checks stop the pipeline, while `WARNING` checks are logged without stopping the run).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.ops.data_quality_results (
    run_id STRING NOT NULL,
    layer STRING NOT NULL,
    check_name STRING NOT NULL,
    severity STRING NOT NULL,
    passed BOOLEAN NOT NULL,
    actual_value DOUBLE,
    message STRING,
    checked_at_utc TIMESTAMP NOT NULL
) USING DELTA
""")

# Check created tables
for schema_name in ["bronze", "silver", "gold", "ops"]:
    print(f"Created or verified schema: {CATALOG}.{schema_name}")